In [1]:
import pandas as pd
import numpy as np

In [2]:
np.random.seed(42)

n = 10000

In [6]:
order_ids = [f"AMZ-IN-{400000+i}" for i in range(n)]
user_ids = [f"CUST-{str(i).zfill(6)}" for i in np.random.randint(1000, 5000, n)]
seller_ids = [f"AMZ-SEL-{str(i).zfill(5)}" for i in np.random.randint(100, 1500, n)]

In [7]:
order_dates = pd.to_datetime(
    np.random.choice(pd.date_range("2024-01-01", "2024-12-31", freq="H"), n))

/tmp/ipykernel_10483/83047318.py:2: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  np.random.choice(pd.date_range("2024-01-01", "2024-12-31", freq="H"), n))


In [10]:
categories = ["Electronics", "Fashion & Apparel", "Home & Kitchen",
              "Books & Media", "Beauty & Personal", "Grocery & Food"]

cities = ["Mumbai", "Delhi", "Bangalore", "Hyderabad", "Chennai",
          "Ahmedabad", "Nagpur", "Gurgaon", "Visakhapatnam"]

city_tier_map = {
    "Mumbai": "Tier 1", "Delhi": "Tier 1", "Bangalore": "Tier 1",
    "Hyderabad": "Tier 1", "Chennai": "Tier 1", "Gurgaon": "Tier 1",
    "Ahmedabad": "Tier 2", "Nagpur": "Tier 2", "Visakhapatnam": "Tier 3"
}

seller_types = ["Reseller", "Brand/Manufacturer", "D2C Brand"]
seller_plans = ["Individual", "Professional"]

payment_methods = ["UPI", "Credit Card", "COD", "Amazon Pay"]

In [11]:
df = pd.DataFrame({
    "order_id": order_ids,
    "order_date": order_dates,
    "user_id": user_ids,
    "is_prime_member": np.random.choice([0,1], n, p=[0.6,0.4]),
    "seller_id": seller_ids,
    "seller_type": np.random.choice(seller_types, n),
    "seller_plan": np.random.choice(seller_plans, n),
    "city": np.random.choice(cities, n),
    "category": np.random.choice(categories, n),
    "quantity": np.random.randint(1,5, n)
})

In [12]:
df["city_tier"] = df["city"].map(city_tier_map)

df["order_value"] = np.round(np.random.uniform(100, 10000, n), 2)

df["fulfillment_type"] = np.random.choice(["FBA", "Merchant"], n)

df["referral_fee"] = df["order_value"] * np.random.uniform(0.05, 0.15, n)
df["fba_fee"] = np.where(df["fulfillment_type"]=="FBA",
                         df["order_value"] * 0.1, 0)

df["closing_fee"] = np.random.uniform(10, 100, n)
df["shipping_fee"] = np.random.uniform(20, 200, n)

df["total_amazon_fees"] = (
    df["referral_fee"] + df["fba_fee"] + df["closing_fee"] + df["shipping_fee"]
)

df["gst_on_fees"] = df["total_amazon_fees"] * 0.18

df["seller_payout"] = df["order_value"] - df["total_amazon_fees"] - df["gst_on_fees"]

df["payment_method"] = np.random.choice(payment_methods, n)

df["payment_status"] = np.where(
    df["payment_method"]=="COD",
    np.random.choice(["success", "failed"], n, p=[0.85,0.15]),
    "success"
)

df["delivery_days"] = np.random.randint(1,8,n)
df["delivery_status"] = "delivered"

df["refund_status"] = np.random.choice(["yes","no"], n, p=[0.1,0.9])
df["refund_amount"] = np.where(df["refund_status"]=="yes",
                               df["order_value"]*np.random.uniform(0.3,1.0,n),
                               0)

In [13]:
df["fraud_flag"] = np.random.choice([0,1], n, p=[0.95,0.05])

leakage_types = ["promo_coupon_abuse", "fake_return", "payment_failure", None]

df["leakage_type"] = np.where(
    df["fraud_flag"]==1,
    np.random.choice(leakage_types[:-1], n),
    None
)

df["leakage_amount"] = np.where(
    df["fraud_flag"]==1,
    df["order_value"] * np.random.uniform(0.1,0.5,n),
    0
)

In [14]:
df.to_csv("amazon_india_leakage_10k_generated.csv", index=False)

df.head()

,order_id,order_date,user_id,is_prime_member,seller_id,seller_type,seller_plan,city,category,quantity,...,seller_payout,payment_method,payment_status,delivery_days,delivery_status,refund_status,refund_amount,fraud_flag,leakage_type,leakage_amount
0,AMZ-IN-400000,2024-12-10 18:00:00,CUST-004962,0,AMZ-SEL-00489,Reseller,Individual,Delhi,Grocery & Food,1,...,6523.925700,COD,failed,1,delivered,no,0.0,0,None,0.0
1,AMZ-IN-400001,2024-07-07 13:00:00,CUST-003578,1,AMZ-SEL-00859,D2C Brand,Individual,Gurgaon,Fashion & Apparel,1,...,6338.975006,Credit Card,success,1,delivered,no,0.0,0,None,0.0
2,AMZ-IN-400002,2024-08-05 01:00:00,CUST-001105,1,AMZ-SEL-00982,Brand/Manufacturer,Individual,Chennai,Books & Media,3,...,5796.367711,COD,success,2,delivered,no,0.0,0,None,0.0
3,AMZ-IN-400003,2024-01-22 08:00:00,CUST-003529,0,AMZ-SEL-01071,Reseller,Professional,Chennai,Electronics,4,...,4344.836013,COD,success,6,delivered,no,0.0,0,None,0.0
4,AMZ-IN-400004,2024-03-21 02:00:00,CUST-001205,1,AMZ-SEL-01194,D2C Brand,Professional,Delhi,Fashion & Apparel,3,...,5390.608814,Credit Card,success,6,delivered,no,0.0,0,None,0.0
